<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 2 — Python Toolkit: Functions as Inputs</div>
</div>

#### The Python behind this week's root-finders and fits: passing functions to functions, lambda, returning several values, default arguments, *args and errors. About 15 minutes; then keep it as a reference.
***

## Why this notebook

This week's root-finders and fitting routines all take **a function as an input**: you hand `bisection` the function whose root you want, and you hand `curve_fit` the model you want to fit. The same pattern appears in every library routine you'll use from now on (`brentq`, `curve_fit`, `minimize`, `solve_ivp`, ...).

This short notebook covers the Python behind that pattern. Work through it first (about 15 minutes), and come back to it as a reference.

## <span style="color:#B5121B">1. Functions are objects</span>

A function is a value like any other: you can store it in a variable, put it in a list, and **pass it to another function**.

In [ ]:
import numpy as np

def evaluate_on_grid(f, x_min, x_max, n=5):
    """Evaluate any function f at n evenly spaced points."""
    x = np.linspace(x_min, x_max, n)
    return f(x)

def square(x):
    return x**2

print(evaluate_on_grid(square, 0, 2))      # pass our own function (no brackets!)
print(evaluate_on_grid(np.sin, 0, np.pi))  # or a NumPy function

Note `square`, not `square()`: with brackets you would *call* the function and pass its result; without them you pass the function itself.

> **Quick check:** write a function `largest_value(f, x_min, x_max)` that returns the maximum of any function on a grid of 1000 points between `x_min` and `x_max`. Test it on `np.cos` between 0 and π (answer: 1).

In [ ]:
# TODO: largest_value(f, x_min, x_max)

## <span style="color:#B5121B">2. Returning several values</span>

A function can return several values at once, as a *tuple*. Unpack them into separate names when you call it.

In [ ]:
def min_and_max(values):
    """Return the smallest and largest values."""
    return np.min(values), np.max(values)

low, high = min_and_max(np.array([3.2, -1.0, 7.5]))
print(f"low = {low}, high = {high}")

result = min_and_max(np.array([3.2, -1.0, 7.5]))   # or keep the tuple
print(result, result[0])

## <span style="color:#B5121B">3. Default and keyword arguments</span>

Arguments can have **default values**, used when the caller doesn't give one. Callers can also name arguments, in any order. This is how `tol=1e-12` and `max_iter=50` work in today's root-finders.

In [ ]:
def decay(t, N0=1000, half_life=5.0):
    """Number of nuclei remaining after time t (same time units as half_life)."""
    return N0 * 0.5 ** (t / half_life)

print(decay(10))                       # both defaults
print(decay(10, half_life=2.0))        # change one, by name
print(decay(t=10, N0=50, half_life=2.0))

## <span style="color:#B5121B">4. `lambda`: small functions in one line</span>

`lambda x: expression` makes a function without a name. It's handy for short functions you need only once, and especially for **fixing some of a function's arguments**.

The root-finders want a function of *one* variable, $f(E)$. But Kepler's equation also depends on $e$ and $M$. A `lambda` makes a one-variable version with $e$ and $M$ fixed:

In [ ]:
def kepler_f(E, e, M):
    return E - e * np.sin(E) - M

e, M = 0.5, 1.0
f = lambda E: kepler_f(E, e, M)      # a function of E only

print(f(1.0), kepler_f(1.0, 0.5, 1.0))   # the same

# Written out in full, the lambda is just:
def f_long(E):
    return kepler_f(E, e, M)
print(f_long(1.0))

> **Quick check:** using `evaluate_on_grid` from Section 1 and a `lambda`, evaluate $x^3 - 2x - 5$ at 5 points between 0 and 3.

In [ ]:
# TODO

## <span style="color:#B5121B">5. Any number of arguments: `*args`</span>

`curve_fit` doesn't know how many parameters your model has, so it calls `model(x, *params)`. The `*` **unpacks** a list or array into separate arguments:

In [ ]:
def line(x, a, b):
    return a + b * x

params = [2.0, 0.5]
print(line(3.0, *params))      # the same as line(3.0, 2.0, 0.5)

In the other direction, `def model(x, *p)` **collects** any number of arguments into a tuple `p`. This lets one function handle any number of parameters, for example a polynomial of any degree:

In [ ]:
def polynomial(x, *coeffs):
    """c0 + c1 x + c2 x^2 + ... for any number of coefficients."""
    return sum(c * x**k for k, c in enumerate(coeffs))

print(polynomial(2.0, 1, 1))        # 1 + 2
print(polynomial(2.0, 1, 1, 1))     # 1 + 2 + 4

## <span style="color:#B5121B">6. Errors: raising and catching them</span>

If a function is given input it can't handle, it should stop with a clear message rather than return something wrong. `raise` does that; `try` / `except` catches the error if you want to carry on.

In [ ]:
def safe_log10(x):
    """Base-10 logarithm, refusing non-positive input."""
    if x <= 0:
        raise ValueError(f"safe_log10 needs x > 0, got {x}")
    return np.log10(x)

print(safe_log10(1000))
try:
    safe_log10(-5)
except ValueError as err:
    print("Caught:", err)

The Week 2 notebooks use this. Your unfinished functions contain `raise NotImplementedError`, and the test cells catch it and print *"Not implemented yet"*, so you can run every cell before you've written everything. Once you fill in a function, the test cell runs it for real.

## Where you'll meet this

| Routine | You pass it | Week |
|---|---|---|
| your `bisection`, `newton` | the function whose root you want | 2 |
| `scipy.optimize.brentq` | the function, and a bracket | 2 |
| `scipy.optimize.curve_fit` | the model `model(x, *params)` | 2 |
| `scipy.optimize.minimize` | the function to minimise | 2 |
| `scipy.integrate.solve_ivp` | the right-hand side of $dy/dt = f(t, y)$ | Lecture 2 |

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>